# RQ-5 — Wie stabil ist der Agent bei wiederholter Ausführung?

Dieses Notebook wertet die Stabilitätsstudie zu Forschungsfrage 5 aus. Untersucht wird, ob der Data-Science-Agent bei mehrfacher Ausführung auf demselben Datensatz konsistent gute Ergebnisse liefert oder ob die Qualität zwischen Läufen stark schwankt.

Versuchsaufbau: 5 Datensätze, jeweils 5 unabhängige Läufe in der besten Konfiguration aus RQ-2 (*EN+Python*). Pro Lauf liegen sowohl SEVQ- als auch VER-Werte vor.

Die Auswertung umfasst:

- deskriptive Stabilitätskennzahlen pro Datensatz (Mittelwert, Standardabweichung, Variationskoeffizient *CV*, Spannweite),
- Intraklassen-Korrelation ICC(2,1) für den SEVQ-Total und für jede der sechs SEVQ-Dimensionen einzeln,
- VER-Stabilität (finale Erfolgsrate pro Lauf),
- Likert-Bewertungen aus den Studienfragen 31 und 32, in denen Teilnehmende die wahrgenommene Konsistenz zweier Datensatzergebnisse beurteilt haben,
- Abbildungen für die Thesis und JSON-Export der Kennzahlen.

Das Notebook ist Teil des Reproduktionspakets der Masterarbeit und kann von oben nach unten ausgeführt werden.


## Setup

Bibliotheken laden. Neben dem üblichen Stack (`pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy.stats`) wird hier zusätzlich `pingouin` verwendet — die Bibliothek liefert eine sauber implementierte Routine für die Intraklassen-Korrelation inklusive Konfidenzintervallen. `matplotlib` läuft im `Agg`-Backend, damit das Notebook auch ohne Display (CI, Server) durchläuft.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import pingouin as pg
from scipy import stats
import json


## Konfiguration und Pfade

Alle festen Werte sind hier gesammelt:

- `METRICS` — die sechs SEVQ-Dimensionen.
- `BASE_PATH` — Wurzelverzeichnis der Stabilitätsläufe. Pro Datensatz existiert ein Unterordner mit fünf `run_*`-Verzeichnissen.
- `STUDY_PATH` — CSV-Export der Online-Studie aus LamaPoll.
- `OUTPUT_DIR` / `THESIS_FIG_DIR` — Zielordner für die generierten Abbildungen. Es wird parallel in den Thesis-Quellbaum geschrieben, damit die Typst-Datei die Plots ohne Umweg einbinden kann.
- `DATASETS` — wird zur Laufzeit aus dem Dateisystem gelesen, sodass die Auswertung auch dann funktioniert, wenn Datensätze nachträglich hinzukommen.
- `N_RUNS` — Anzahl Wiederholungen pro Datensatz.
- `LIKERT_MAP` — Übersetzung der deutschen LamaPoll-Antworttexte auf die Skala 1–5.
- `DIMS_DE` / `DIMS_DE_FLAT` — deutsche Dimensionsnamen für die Likert-Auswertung (zwei Varianten: mit Zeilenumbruch für Achsenbeschriftungen, ohne für die Dictionary-Schlüssel).
- `PALETTE_RUNS` — wird sowohl für Datensätze (im Liniendiagramm) als auch für Läufe genutzt.


In [ ]:
METRICS = ['bugs', 'transformation', 'compliance', 'type', 'encoding', 'aesthetics']
MODEL_MAPPING = {
    'gpt-5': 'GPT-5',
    'gpt-4o': 'GPT-4o',
    'google/gemini-3-flash-preview': 'Gemini',
    'x-ai/grok-4.1-fast': 'Grok',
    'anthropic/claude-sonnet-4.5': 'Claude Sonnet'
}

BASE_PATH = Path("./src/resources/paper_results/continous_quality/output")
STUDY_PATH = Path("./results/rq_5/study_data.csv")
OUTPUT_DIR = Path("./results/rq_5/figures")
THESIS_FIG_DIR = OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
THESIS_FIG_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = sorted([d.name for d in BASE_PATH.iterdir() if d.is_dir()])
N_RUNS = 5

LIKERT_MAP = {
    'Stimme voll und ganz zu': 5,
    'Stimme zu': 4,
    'Stimme weder zu noch lehne ich ab': 3,
    'Stimme nicht zu': 2,
    'Stimme überhaupt nicht zu': 1,
}

DIMS_DE = ['Klarheit', 'Aussagekraft /\nRelevanz', 'Gestalterische\nQualität',
           'Informations-\ngehalt', 'Interpretier-\nbarkeit', 'Plausibilität /\nKorrektheit']
DIMS_DE_FLAT = ['Klarheit', 'Aussagekraft / Relevanz', 'Gestalterische Qualität / Layout',
                'Informationsgehalt', 'Interpretierbarkeit', 'Plausibilität / Korrektheit']

sns.set_theme(style='whitegrid', context='talk', font='DejaVu Sans')
PALETTE_RUNS = sns.color_palette('Set2', N_RUNS)


## Daten laden

Drei kleine Loader-Funktionen:

- `load_sevq_run` liest die `sevq.csv` eines bestimmten Laufs eines bestimmten Datensatzes ein und ergänzt `dataset_id` und `run` als Spalten, damit die Datensätze später zu einem langen DataFrame zusammengeführt werden können.
- `load_ver_final` lädt die VER-Datei mit der höchsten Iterationsnummer (`VER_*.csv`). Da der Agent iterativ verbessert, ist das die *finale* Erfolgsrate eines Laufs. Rückgabe in Prozent.
- `compute_sevq_total` summiert die sechs SEVQ-Dimensionen je Bewertungszeile und mittelt anschließend über alle Bewertungen — das entspricht dem SEVQ-Total eines Laufs.


In [ ]:
def load_sevq_run(dataset_id: str, run_i: int) -> pd.DataFrame:
    """Lädt die SEVQ-Daten eines bestimmten Datensatzes und Laufs."""
    sevq_f = BASE_PATH / dataset_id / f"run_{run_i}" / 'sevq.csv'
    if not sevq_f.exists():
        return pd.DataFrame()
    df = pd.read_csv(sevq_f)
    df['dataset_id'] = dataset_id
    df['run'] = run_i
    return df


def load_ver_final(dataset_id: str, run_i: int) -> float:
    """Lädt die finale VER (höchste Iteration) für einen Datensatz und Lauf."""
    run_dir = BASE_PATH / dataset_id / f"run_{run_i}"
    ver_files = sorted(run_dir.glob('VER_*.csv'))
    if not ver_files:
        return np.nan
    last_ver = pd.read_csv(ver_files[-1])
    return last_ver['ver_value'].mean() * 100


def compute_sevq_total(df: pd.DataFrame) -> float:
    """Mittlerer SEVQ-Total über alle Judges und Visualisierungen."""
    return df[METRICS].sum(axis=1).mean()


## SEVQ und VER pro Lauf einsammeln

Iteration über alle Datensätze und alle fünf Läufe. Pro Lauf werden vier Dinge berechnet:

1. der SEVQ-Total,
2. die finale VER,
3. die Mittelwerte der sechs SEVQ-Einzeldimensionen,
4. plus die Identifier (`dataset_id`, `run`).

Das Ergebnis ist ein langer DataFrame `df_results` mit einer Zeile pro (Datensatz, Lauf)-Kombination — die zentrale Datenstruktur für alle weiteren Schritte.


In [ ]:
print("=" * 60)
print("RQ-5: Ergebnisstabilität")
print("=" * 60)

results = []
for ds in DATASETS:
    for run_i in range(1, N_RUNS + 1):
        df = load_sevq_run(ds, run_i)
        if df.empty:
            continue
        sevq_total = compute_sevq_total(df)
        ver_final = load_ver_final(ds, run_i)
        dim_means = {m: df[m].mean() for m in METRICS}
        results.append({
            'dataset_id': ds,
            'run': run_i,
            'sevq_total': sevq_total,
            'ver_final': ver_final,
            **dim_means
        })

df_results = pd.DataFrame(results)
df_results.head()


## Stabilitätskennzahlen pro Datensatz

Aus den fünf Läufen pro Datensatz werden die klassischen Streuungskennzahlen berechnet:

- **Mittelwert** und **Standardabweichung** des SEVQ-Total,
- **Variationskoeffizient (CV)** in Prozent — die SD relativ zum Mittelwert. Der CV erlaubt einen fairen Vergleich der Streuung zwischen Datensätzen mit unterschiedlichem Niveau,
- **Range** und **Min/Max** des SEVQ-Total,
- **Mittelwert** und **SD** der VER,
- ein Indikator `ver_identical_runs`, der genau dann 1 ist, wenn die VER über alle fünf Läufe identisch war.

Anschließend werden die globalen Mittelwerte (über alle Datensätze) ausgegeben — diese drei Zahlen sind die Hauptkennzahlen der SEVQ-Stabilität in der Thesis.


In [ ]:
stability = []
for ds in DATASETS:
    print(df_results.head(10))
    ds_data = df_results[df_results['dataset_id'] == ds]
    sevq_vals = ds_data['sevq_total'].values
    ver_vals = ds_data['ver_final'].values

    cv_sevq = stats.variation(sevq_vals, ddof=1) * 100  # CV in %

    stability.append({
        'dataset_id': ds,
        'sevq_mean': np.mean(sevq_vals),
        'sevq_sd': np.std(sevq_vals, ddof=1),
        'sevq_cv': cv_sevq,
        'sevq_range': np.ptp(sevq_vals),
        'sevq_min': np.min(sevq_vals),
        'sevq_max': np.max(sevq_vals),
        'ver_mean': np.mean(ver_vals),
        'ver_sd': np.std(ver_vals, ddof=1),
        'ver_identical_runs': int(np.sum(ver_vals == ver_vals[0]) == len(ver_vals)),
    })

df_stability = pd.DataFrame(stability)

print("\n--- SEVQ-Total Stabilität pro Datensatz ---")
print(df_stability[['dataset_id', 'sevq_mean', 'sevq_sd', 'sevq_cv', 'sevq_range']].to_string(index=False))

global_cv = df_stability['sevq_cv'].mean()
global_sd = df_stability['sevq_sd'].mean()
global_range = df_stability['sevq_range'].mean()
print(f"\nGlobal: Mean CV = {global_cv:.2f}%, Mean SD = {global_sd:.2f}, Mean Range = {global_range:.2f}")

print("\n--- VER Stabilität pro Datensatz ---")
print(df_stability[['dataset_id', 'ver_mean', 'ver_sd', 'ver_identical_runs']].to_string(index=False))


## Intraklassen-Korrelation ICC(2,1) für den SEVQ-Total

Während CV und SD die Streuung *innerhalb* eines Datensatzes beschreiben, beantwortet die Intraklassen-Korrelation die übergeordnete Frage: *Wie groß ist der Anteil der Gesamtvarianz, der auf systematische Unterschiede zwischen Datensätzen zurückgeht — im Vergleich zur zufälligen Schwankung zwischen Läufen?*

Verwendet wird ICC(2,1) im Sinne von Shrout & Fleiss: ein Two-Way-Random-Modell, bei dem sowohl Datensätze (*targets*) als auch Läufe (*raters*) als zufällig betrachtet werden, und die Zuverlässigkeit eines *einzelnen* Laufs quantifiziert wird. Hohe Werte (> 0.75) sprechen für eine gute Reproduzierbarkeit, niedrige Werte für hohe Run-zu-Run-Varianz.

Die Routine `pingouin.intraclass_corr` liefert eine Tabelle mit allen ICC-Varianten — für RQ-5 wird die Zeile `ICC2` ausgewertet (dort stehen Punktschätzer, 95%-KI und p-Wert).


In [ ]:
# Long-Format für pingouin: targets = Datensätze, raters = Runs
icc_long = df_results[['dataset_id', 'run', 'sevq_total']].copy()
icc_long.rename(columns={'dataset_id': 'targets', 'run': 'raters', 'sevq_total': 'ratings'}, inplace=True)

icc_table = pg.intraclass_corr(
    data=icc_long, targets='targets', raters='raters', ratings='ratings'
)
icc_row = icc_table[icc_table['Type'] == 'ICC2']
icc_value = icc_row['ICC'].values[0]
icc_ci_low = icc_row['CI95%'].values[0][0]
icc_ci_high = icc_row['CI95%'].values[0][1]
icc_pval = icc_row['pval'].values[0]

print(f"ICC(2,1) über Runs (SEVQ-Total): {icc_value:.3f}  "
      f"[95% CI: {icc_ci_low:.3f}–{icc_ci_high:.3f}], p = {icc_pval:.4f}")


## ICC(2,1) pro SEVQ-Dimension

Derselbe Test wird zusätzlich für jede der sechs SEVQ-Dimensionen einzeln gerechnet. So lässt sich erkennen, ob bestimmte Dimensionen (z.B. *aesthetics*) systematisch instabiler sind als andere (z.B. *bugs*). Die einzelnen ICC-Werte fließen sowohl in die Diskussion als auch in die JSON-Zusammenfassung ein.


In [ ]:
print("--- ICC(2,1) pro SEVQ-Dimension ---")
dim_iccs = {}
for m in METRICS:
    dim_long = df_results[['dataset_id', 'run']].copy()
    dim_long['ratings'] = df_results[m]
    dim_long.rename(columns={'dataset_id': 'targets', 'run': 'raters'}, inplace=True)
    dim_icc_table = pg.intraclass_corr(
        data=dim_long, targets='targets', raters='raters', ratings='ratings'
    )
    dim_row = dim_icc_table[dim_icc_table['Type'] == 'ICC2']
    dim_icc_val = dim_row['ICC'].values[0]
    dim_ci = dim_row['CI95%'].values[0]
    dim_iccs[m] = dim_icc_val
    print(f"  {m}: ICC = {dim_icc_val:.3f}  [95% CI: {dim_ci[0]:.3f}–{dim_ci[1]:.3f}]")


## Likert-Auswertung der Studienfragen 31 und 32

In den Fragen 31 und 32 der Online-Studie wurden die Teilnehmenden gebeten, die Konsistenz der Visualisierungen zweier exemplarischer Datensätze auf einer Likert-Skala zu bewerten — entlang derselben sechs Dimensionen, die auch im SEVQ verwendet werden. Frage 31 bezieht sich auf den ersten Beispieldatensatz, Frage 32 auf den zweiten.

### Spaltensuche

Die exportierten Spaltennamen sind lang und enthalten die Dimensionen als Substring. Statt die exakten Namen hartzukodieren (was bei kleinsten Änderungen am Studienexport bricht), wird über ein Stichwort (`dim_search`) das passende Spaltenlabel gefunden. Spalten mit `Punkte` oder `Punktesumme` im Namen werden ausgeschlossen — das sind die von LamaPoll automatisch berechneten Aggregate, nicht die Einzelantworten.

### Mapping auf 1–5

Die deutschen Antworttexte werden über `LIKERT_MAP` auf die numerische Skala 1–5 abgebildet. Anschließend werden M, SD und Median pro Dimension berichtet.


In [ ]:
print("=" * 60)
print("Likert-Analyse (Studienteilnehmende)")
print("=" * 60)

df_study = pd.read_csv(STUDY_PATH, sep=';')

dims_search = ['Klarheit', 'Aussagekraft', 'Gestalterische',
               'Informationsgehalt', 'Interpretierbarkeit', 'Plausibilität']

likert_data = {}
for q in [31, 32]:
    q_data = {}
    for dim_s, dim_full in zip(dims_search, DIMS_DE_FLAT):
        col = [c for c in df_study.columns
               if f"Frage {q}" in c and dim_s in c
               and "Punkte" not in c and "Punktesumme" not in c]
        if col:
            values = df_study[col[0]].map(LIKERT_MAP)
            q_data[dim_full] = values.tolist()
    likert_data[f'Q{q}'] = q_data

for q_label, q_data in likert_data.items():
    print(f"\n--- {q_label} ---")
    for dim, vals in q_data.items():
        arr = np.array(vals)
        print(f"  {dim}: M={arr.mean():.2f}, SD={arr.std(ddof=1):.2f}, Md={np.median(arr):.1f}")


## Visualisierungen für die Thesis

Im Folgenden werden fünf Abbildungen erzeugt. Jede Abbildung wird über den Helfer `_save` als SVG (für Typst) und PNG (für die Vorschau) abgelegt — und zwar parallel in `OUTPUT_DIR` und in den Thesis-Quellbaum, damit der Build der Thesis ohne Kopieraktion funktioniert.

`dataset_labels` ersetzt die langen GovData-IDs durch kompakte Kurzbezeichner (`DS 1`, `DS 2`, …) für die Achsenbeschriftungen.


In [ ]:
dataset_labels = {ds: f'DS {i+1}' for i, ds in enumerate(DATASETS)}


def _save(fig, name):
    """Speichert die Abbildung als SVG und PNG in beide Zielordner."""
    for d in [OUTPUT_DIR, THESIS_FIG_DIR]:
        fig.savefig(d / f'{name}.svg', bbox_inches='tight')
        fig.savefig(d / f'{name}.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"[saved] {name}.svg")


### Abbildung 1 — SEVQ-Total pro Lauf, je Datensatz

Liniendiagramm: x-Achse sind die fünf Läufe, y-Achse der SEVQ-Total. Pro Datensatz eine Linie. Die gestrichelten waagrechten Linien markieren den Mittelwert je Datensatz und machen optisch sichtbar, wie weit einzelne Läufe vom Datensatz-Mittel abweichen.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for i, ds in enumerate(DATASETS):
    ds_data = df_results[df_results['dataset_id'] == ds].sort_values('run')
    ax.plot(ds_data['run'], ds_data['sevq_total'], 'o-',
            color=PALETTE_RUNS[i], label=dataset_labels[ds],
            markersize=8, linewidth=2)
    mean_val = ds_data['sevq_total'].mean()
    ax.axhline(y=mean_val, color=PALETTE_RUNS[i], linestyle='--', alpha=0.3, linewidth=1)

ax.set_xlabel('Run')
ax.set_ylabel('SEVQ-Total (0–60)')
ax.set_xticks(range(1, N_RUNS + 1))
ax.legend(loc='lower right', fontsize=11)
ax.set_ylim(40, 56)
ax.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
_save(fig, '05-rq5-sevq-total-per-run')


### Abbildung 2 — Variationskoeffizient pro Datensatz

Balkendiagramm des CV (in Prozent) je Datensatz. Die CV-Werte stehen direkt über den Balken. Die Farben greifen die Datensatz-Palette aus dem ersten Plot wieder auf, damit Datensätze über die Abbildungen hinweg wiedererkennbar bleiben.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(range(len(DATASETS)), df_stability['sevq_cv'].values,
              color=[PALETTE_RUNS[i] for i in range(len(DATASETS))],
              edgecolor='black', linewidth=0.5)
for bar, cv in zip(bars, df_stability['sevq_cv'].values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'{cv:.1f}%', ha='center', va='bottom', fontsize=11)
ax.set_xticks(range(len(DATASETS)))
ax.set_xticklabels([dataset_labels[ds] for ds in DATASETS])
ax.set_ylabel('Variationskoeffizient CV (%)')
ax.set_xlabel('Datensatz')
ax.set_ylim(0, max(df_stability['sevq_cv'].values) * 1.3)
ax.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
_save(fig, '05-rq5-sevq-cv-per-dataset')


### Abbildung 3 — VER pro Lauf, je Datensatz

Analog zu Abbildung 1, aber für die finale VER. Die Marker sind hier Quadrate, damit der Plot visuell klar von Abbildung 1 unterscheidbar bleibt. Die y-Achse ist auf den Bereich –5 bis 110 % aufgespannt, damit Läufe mit 0 % oder 100 % an den Achsenrändern nicht abgeschnitten werden.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for i, ds in enumerate(DATASETS):
    ds_data = df_results[df_results['dataset_id'] == ds].sort_values('run')
    ax.plot(ds_data['run'], ds_data['ver_final'], 's-',
            color=PALETTE_RUNS[i], label=dataset_labels[ds],
            markersize=8, linewidth=2)
ax.set_xlabel('Run')
ax.set_ylabel('VER (%)')
ax.set_xticks(range(1, N_RUNS + 1))
ax.set_ylim(-5, 110)
ax.legend(loc='lower right', fontsize=11)
ax.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
_save(fig, '05-rq5-ver-per-run')


### Abbildung 4 — Likert-Bewertungen als Diverging Stacked Bar Chart

Visualisierung der Likert-Antworten zu Frage 31 (linkes Panel, Datensatz 1) und Frage 32 (rechtes Panel, Datensatz 2). Pro Dimension wird ein gestapelter horizontaler Balken erzeugt, der nach links die ablehnenden Stimmen (1, 2) und nach rechts die zustimmenden Stimmen (4, 5) abträgt. Die neutrale Kategorie (3) ist symmetrisch um die Null gelegt, sodass die Nulllinie weiterhin als visueller Anker für „Zustimmung vs. Ablehnung“ funktioniert.

`n_total = 14` ist die Anzahl der Studienteilnehmenden; die Balkenlängen sind jeweils Anteile in Prozent davon.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

likert_colors = {1: '#334152', 2: '#546B86', 3: '#B2BFCE', 4: '#D9E5EC', 5: '#009B91'}

for ax_idx, (q_label, q_data) in enumerate(likert_data.items()):
    ax = axes[ax_idx]
    dim_names = list(q_data.keys())
    n_dims = len(dim_names)
    n_total = 14

    for dim_i, dim in enumerate(dim_names):
        arr = np.array(q_data[dim])
        counts = {v: int(np.sum(arr == v)) for v in [1, 2, 3, 4, 5]}

        # Negative (links)
        left_2 = -counts[2] / n_total * 100
        left_1 = -(counts[1] + counts[2]) / n_total * 100
        # Positive (rechts)
        right_4 = counts[4] / n_total * 100
        right_5 = (counts[4] + counts[5]) / n_total * 100
        # Neutral (zentriert um die Null)
        neutral = counts[3] / n_total * 100

        ax.barh(dim_i, left_1 - left_2, left=left_2, color=likert_colors[1], height=0.6, edgecolor='white', linewidth=0.5)
        ax.barh(dim_i, left_2, left=0, color=likert_colors[2], height=0.6, edgecolor='white', linewidth=0.5)
        ax.barh(dim_i, -neutral / 2, left=0, color=likert_colors[3], height=0.6, edgecolor='white', linewidth=0.5)
        ax.barh(dim_i, neutral / 2, left=0, color=likert_colors[3], height=0.6, edgecolor='white', linewidth=0.5)
        ax.barh(dim_i, right_4, left=0, color=likert_colors[4], height=0.6, edgecolor='white', linewidth=0.5)
        ax.barh(dim_i, right_5 - right_4, left=right_4, color=likert_colors[5], height=0.6, edgecolor='white', linewidth=0.5)

    ax.set_yticks(range(n_dims))
    ax.set_yticklabels([DIMS_DE[i] for i in range(n_dims)], fontsize=10)
    ax.set_xlabel('Anteil (%)')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-105, 105)
    q_title = 'Datensatz 1' if q_label == 'Q31' else 'Datensatz 2'
    ax.set_title(q_title, fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.2)

legend_elements = [
    Patch(facecolor=likert_colors[1], label='Stimme überhaupt nicht zu'),
    Patch(facecolor=likert_colors[2], label='Stimme nicht zu'),
    Patch(facecolor=likert_colors[3], label='Neutral'),
    Patch(facecolor=likert_colors[4], label='Stimme zu'),
    Patch(facecolor=likert_colors[5], label='Stimme voll und ganz zu'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))
sns.despine()
plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
_save(fig, '05-rq5-likert-stability')


### Abbildung 5 — Heatmap der SD pro Dimension und Datensatz

Heatmap mit Datensätzen auf der y-Achse und SEVQ-Dimensionen auf der x-Achse. Die Zellwerte sind die Standardabweichungen der jeweiligen Dimension über die fünf Läufe. So ist auf einen Blick erkennbar, *welche* Dimension *welchen* Datensatz instabil macht — eine deutlich detailliertere Sicht als der globale CV.

Die Schriftfarbe der Zellbeschriftung wechselt automatisch von Schwarz auf Weiß, sobald die Hintergrundfarbe zu dunkel wird (Schwellwert: 60 % des Maximalwerts), damit die Annotationen lesbar bleiben.


In [ ]:
dim_sd_matrix = np.zeros((len(DATASETS), len(METRICS)))
for i, ds in enumerate(DATASETS):
    ds_data = df_results[df_results['dataset_id'] == ds]
    for j, m in enumerate(METRICS):
        dim_sd_matrix[i, j] = ds_data[m].std(ddof=1)

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(dim_sd_matrix, cmap='YlOrRd', aspect='auto', vmin=0)
ax.set_xticks(range(len(METRICS)))
ax.set_xticklabels([m.capitalize() for m in METRICS], fontsize=11)
ax.set_yticks(range(len(DATASETS)))
ax.set_yticklabels([dataset_labels[ds] for ds in DATASETS], fontsize=11)

for i in range(len(DATASETS)):
    for j in range(len(METRICS)):
        val = dim_sd_matrix[i, j]
        color = 'white' if val > dim_sd_matrix.max() * 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10, color=color)

plt.colorbar(im, ax=ax, label='SD über 5 Runs', shrink=0.9)
ax.set_xlabel('SEVQ-Dimension')
ax.set_ylabel('Datensatz')
plt.tight_layout()
_save(fig, '05-rq5-dimension-sd-heatmap')


## Export der Kennzahlen als JSON

Alle berechneten Kennzahlen werden in eine `rq5_summary.json` geschrieben:

- globale SEVQ-Stabilität (Mean CV, Mean SD, Mean Range, ICC),
- Stabilitätskennzahlen pro Datensatz,
- ICC pro SEVQ-Dimension,
- VER-Stabilität pro Datensatz,
- Likert-Verteilungen pro Frage und Dimension inkl. globaler Mittelwert pro Frage.

Diese JSON-Datei wird vom Typst-Quelltext der Thesis konsumiert, sodass die Zahlen im Fließtext immer mit dem Notebook synchron sind. Der `default=str`-Fallback fängt NumPy-Typen ab, die `json` sonst nicht serialisieren würde.


In [ ]:
summary = {
    'n_datasets': len(DATASETS),
    'n_runs': N_RUNS,
    'datasets': DATASETS,
    'sevq_stability': {
        'global_mean_cv': round(global_cv, 2),
        'global_mean_sd': round(global_sd, 2),
        'global_mean_range': round(global_range, 2),
        'icc_2_1': round(icc_value, 3),
        'icc_2_1_ci95': [round(icc_ci_low, 3), round(icc_ci_high, 3)],
        'icc_2_1_pval': round(icc_pval, 4),
        'per_dataset': df_stability.to_dict(orient='records'),
    },
    'dimension_iccs': {m: round(v, 3) for m, v in dim_iccs.items()},
    'ver_stability': {
        ds: {
            'mean': round(df_stability.loc[df_stability['dataset_id'] == ds, 'ver_mean'].values[0], 1),
            'sd': round(df_stability.loc[df_stability['dataset_id'] == ds, 'ver_sd'].values[0], 1),
        }
        for ds in DATASETS
    },
    'likert': {},
}

for q_label, q_data in likert_data.items():
    q_summary = {}
    for dim, vals in q_data.items():
        arr = np.array(vals, dtype=float)
        q_summary[dim] = {
            'mean': round(float(np.nanmean(arr)), 2),
            'sd': round(float(np.nanstd(arr, ddof=1)), 2),
            'median': float(np.nanmedian(arr)),
            'distribution': {str(k): int(np.sum(arr == k)) for k in [1, 2, 3, 4, 5]}
        }
    summary['likert'][q_label] = q_summary

for q_label in ['Q31', 'Q32']:
    all_vals = []
    for dim, vals in likert_data[q_label].items():
        all_vals.extend(vals)
    arr = np.array(all_vals, dtype=float)
    summary['likert'][f'{q_label}_global'] = {
        'mean': round(float(np.nanmean(arr)), 2),
        'sd': round(float(np.nanstd(arr, ddof=1)), 2),
        'median': float(np.nanmedian(arr)),
    }

with open(OUTPUT_DIR / 'rq5_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print("[saved] rq5_summary.json")


## Zusammenfassung für die Thesis

Konsolenausgabe der wichtigsten Kennzahlen in einem für die Thesis lesbaren Format. Diese Block dient gleichzeitig als Reproduktionsnachweis: alles, was im Text der Arbeit als Zahl auftaucht, lässt sich hier eins zu eins ablesen.


In [ ]:
print("=" * 60)
print("ZUSAMMENFASSUNG FÜR TYPST")
print("=" * 60)

print(f"\nSEVQ-Total:")
print(f"  Globaler CV: {global_cv:.2f}%")
print(f"  Globale SD: {global_sd:.2f}")
print(f"  Globale Range: {global_range:.2f}")
print(f"  ICC(2,1): {icc_value:.3f}  [95% CI: {icc_ci_low:.3f}–{icc_ci_high:.3f}], p = {icc_pval:.4f}")

print(f"\nPro Datensatz:")
for _, row in df_stability.iterrows():
    print(f"  {dataset_labels[row['dataset_id']]}: M={row['sevq_mean']:.1f}, "
          f"SD={row['sevq_sd']:.2f}, CV={row['sevq_cv']:.1f}%, Range={row['sevq_range']:.1f}")

print(f"\nVER:")
for _, row in df_stability.iterrows():
    print(f"  {dataset_labels[row['dataset_id']]}: M={row['ver_mean']:.1f}%, SD={row['ver_sd']:.1f}%")

print(f"\nDimension ICCs:")
for m, v in dim_iccs.items():
    print(f"  {m}: {v:.3f}")

print(f"\nLikert (Q31 = Datensatz 1):")
for dim, vals in likert_data['Q31'].items():
    arr = np.array(vals)
    print(f"  {dim}: M={arr.mean():.2f}")
print(f"  Gesamt: M={summary['likert']['Q31_global']['mean']}")

print(f"\nLikert (Q32 = Datensatz 2):")
for dim, vals in likert_data['Q32'].items():
    arr = np.array(vals)
    print(f"  {dim}: M={arr.mean():.2f}")
print(f"  Gesamt: M={summary['likert']['Q32_global']['mean']}")

print("\nDone.")
